In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df=pd.read_csv('Dataset.csv').head(1000)

print(df.shape)
print(df.info())
print(df.describe(include='all'))

print("\nMissing Values\n",df.isnull().sum())
print("\nDuplicates:",df.duplicated().sum())

df=df.drop_duplicates()
for c in df.select_dtypes(include='number'):
    df[c]=df[c].fillna(df[c].median())
for c in df.select_dtypes(exclude='number'):
    df[c]=df[c].fillna(df[c].mode()[0])

df["AI_Usage_Level"]=pd.cut(df["Daily_AI_Usage_Hours"],bins=[0,2,5,10],labels=["Low","Medium","High"])

num=df.select_dtypes(include='number')
cat=df.select_dtypes(exclude='number')

plt.figure(figsize=(6,4));sns.histplot(df["Age"],kde=True);plt.title("Age Distribution");plt.show()
plt.figure(figsize=(7,4));sns.countplot(data=df,y="Industry",order=df["Industry"].value_counts().index);plt.title("Industry");plt.show()
plt.figure(figsize=(6,4));sns.boxplot(x="Salary_USD",data=df);plt.show()
plt.figure(figsize=(6,5));sns.heatmap(num.corr(),cmap="Blues");plt.title("Correlation");plt.show()
plt.figure(figsize=(6,4));sns.scatterplot(data=df,x="Daily_AI_Usage_Hours",y="Productivity_Change");plt.show()

fig=px.pie(df,names="Primary_AI_Tool",title="Primary AI Tool")
fig.show()

stats=num.agg(["mean","median","std","min","max"]).T
print(stats)

cluster_cols=["Daily_AI_Usage_Hours","Tasks_Automated_Percent","Time_Saved_Per_Day_Minutes","Productivity_Change","Job_Satisfaction","Salary_USD"]
X=StandardScaler().fit_transform(df[cluster_cols])
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
df["Cluster"]=kmeans.fit_predict(X)
pca=PCA(n_components=2)
pc=pca.fit_transform(X)
plt.figure(figsize=(6,4))
plt.scatter(pc[:,0],pc[:,1],c=df["Cluster"])
plt.title("KMeans Clusters")
plt.show()
print(df.groupby("Cluster")[cluster_cols].mean())

le=LabelEncoder()
data=df.copy()
for c in data.select_dtypes(exclude='number'):
    data[c]=le.fit_transform(data[c].astype(str))

X=data.drop("Target",axis=1)
y=data["Target"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=RandomForestClassifier(random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,pred))
print(confusion_matrix(y_test,pred))

imp=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7,4))
imp.plot(kind="bar")
plt.title("Top Feature Importance")
plt.show()

print("Business Insight: Higher AI usage and automation generally improve productivity and time savings. Industries with greater AI adoption show better overall performance.")


: 